<a href="https://colab.research.google.com/github/juliawol/WB_Sufficiency/blob/main/Notebooks/WB_Final_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 16.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [54]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.metrics import precision_recall_fscore_support

data = pd.read_csv('/content/qa_dataset_labeled.csv')

train_data, val_data = train_test_split(data, test_size=0.2, random_state=42, stratify=data["label"])

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("ai-forever/ruBERT-base")

# Tokenize data
def tokenize(batch):
    return tokenizer(batch["Question"], batch["Description"], truncation=True, padding=True, max_length=512)

train_dataset = Dataset.from_pandas(train_data).map(tokenize, batched=True)
val_dataset = Dataset.from_pandas(val_data).map(tokenize, batched=True)

# Set the format for PyTorch tensors
train_dataset.set_format("torch")
val_dataset.set_format("torch")

# Load pre-trained model
model = AutoModelForSequenceClassification.from_pretrained("ai-forever/ruBERT-base", num_labels=2)

# Define Focal Loss
class FocalLoss(torch.nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        # Convert logits to probabilities
        probs = torch.nn.functional.softmax(logits, dim=-1)
        targets_one_hot = torch.nn.functional.one_hot(targets, num_classes=logits.size(-1)).float()

        # Compute BCE loss
        bce_loss = -(targets_one_hot * torch.log(probs + 1e-6)).sum(dim=-1)

        # Compute Focal Loss adjustment
        pt = torch.exp(-bce_loss)  # Probabilities of correct predictions
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss

        return focal_loss.mean()

# Trainer to use Focal Loss
class FocalLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels").long()
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = FocalLoss(alpha=0.25, gamma=2.0)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    save_total_limit=1,
)

# Compute metrics
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = predictions.argmax(axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {"precision": precision, "recall": recall, "f1": f1}

# Initialize Trainer
trainer = FocalLossTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# Train the model
trainer.train()

# Save the fine-tuned model
model.save_pretrained("./fine_tuned_model")
tokenizer.save_pretrained("./fine_tuned_model")

print("Model fine-tuned and saved successfully!")


Map:   0%|          | 0/309 [00:00<?, ? examples/s]

Map:   0%|          | 0/78 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at ai-forever/ruBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-54-6772c9f90ddc>:87: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `FocalLossTrainer.__init__`. Use `processing_class` instead.
  trainer = FocalLossTrainer(


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.032100,0.033418,0.000000,0.000000,0.000000
2,0.025200,0.028775,0.636364,0.411765,0.500000
3,0.012800,0.030688,0.545455,0.352941,0.428571
4,0.013000,0.038391,0.555556,0.294118,0.384615
5,0.004900,0.040613,0.636364,0.411765,0.500000


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Model fine-tuned and saved successfully!


In [1]:
!pip install --upgrade chromadb


In [2]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [1]:
import chromadb

client = chromadb.PersistentClient(path="./chroma_db")

In [3]:
client = chromadb.EphemeralClient()

In [5]:
data = data.drop_duplicates(subset="NmId")

In [6]:
model_path = "/content/fine_tuned_model"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model =  AutoModelForSequenceClassification.from_pretrained(model_path)

# Function to generate embeddings
def generate_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        embedding = outputs.hidden_states[-1][:, 0, :].squeeze().numpy()  # CLS token embedding
    return embedding

In [7]:
collection = client.get_or_create_collection(name="my_collection")


In [8]:

id_to_description = {row["NmId"]: row["Description"] for _, row in data.iterrows()}

for _, row in data.iterrows():
    embedding = generate_embedding(row["Description"])
    collection.add(
        embeddings=[embedding],
        metadatas=[{"id": str(row["NmId"]), "description": row["Description"]}],
        ids=[str(row["NmId"])]
    )
print("Data added to ChromaDB.")


Data added to ChromaDB.


In [9]:
def retrieve_description_by_id(product_id, collection):
    # Query ChromaDB to retrieve metadata directly by ID
    results = collection.get(ids=[str(product_id)])

    if not results or "metadatas" not in results or len(results["metadatas"]) == 0:
        return None

    # Extract the description from the metadata
    description = results["metadatas"][0].get("description")
    return description


In [10]:
def inference_pipeline(product_id, question, collection, model, tokenizer):
    # Retrieve the product description by ID
    description = retrieve_description_by_id(product_id, collection)

    if not description:
        return {"error": "No description found for the provided product ID."}

    # Tokenize the question and description
    inputs = tokenizer(question, description, return_tensors="pt", truncation=True, padding=True, max_length=512)

    # Predict using the fine-tuned model
    outputs = model(**inputs)
    probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)
    label = torch.argmax(probabilities).item()
    confidence = probabilities[0][label].item()

    return {"label": label, "confidence": confidence, "description": description}


In [11]:
product_id = int(input("Enter Product ID: "))
question = input("Enter Question: ")

result = inference_pipeline(product_id, question, collection, model, tokenizer)
if "error" in result:
    print(result["error"])
else:
    print(f"Label: {result['label']} (Confidence: {result['confidence']:.2f})")
    print(f"Description: {result['description']}")

Enter Product ID: 3440033
Enter Question: Откуда такая цена?
Label: 0 (Confidence: 0.67)
Description: {'Страна производства': 'Таиланд', 'ТНВЭД': '6403511500', 'Метод крепления подошвы': 'Вулканизация', 'Высота подошвы': '1 см', 'Коллекция': 'Осень-Зима 2024', 'Длина упаковки': '34 см', 'Ортопедия': 'нет', 'Материал стельки': 'Искусственная шерсть', 'Ставка НДС': '20', 'Материал подошвы обуви': 'Полиуретан', 'Высота обуви': 'высокие', 'Материал подкладки обуви': 'Искусственная шерсть', 'Дата окончания действия сертификата/декларации': '25.05.2025', 'Высота упаковки': '13 см', 'Назначение обуви': 'повседневная', 'Цвет': 'черный', 'Особенности модели': 'Gore-Tex мембрана', 'Модель ботинок': 'ботинки', 'Дата регистрации сертификата/декларации': '26.05.2022', 'Декоративные элементы': 'без элементов', 'Номер сертификата соответствия': 'ЕАЭС RU С-DK.АЯ46.В.25234/22', 'Комплектация': 'Ботинки - 1 пара', 'Вид застежки': 'Шнурки', 'Полнота обуви (EUR)': 'F (6)', 'Ширина упаковки': '34 см', 'Пол

In [12]:
product_id = int(input("Enter Product ID: "))
question = input("Enter Question: ")

print(f"Label: {result['label']} (Confidence: {result['confidence']:.2f})")
print(f"Description: {result['description']}")

Enter Product ID: 2025564
Enter Question:  А какой состав?
Label: 0 (Confidence: 0.67)
Description: {'Страна производства': 'Таиланд', 'ТНВЭД': '6403511500', 'Метод крепления подошвы': 'Вулканизация', 'Высота подошвы': '1 см', 'Коллекция': 'Осень-Зима 2024', 'Длина упаковки': '34 см', 'Ортопедия': 'нет', 'Материал стельки': 'Искусственная шерсть', 'Ставка НДС': '20', 'Материал подошвы обуви': 'Полиуретан', 'Высота обуви': 'высокие', 'Материал подкладки обуви': 'Искусственная шерсть', 'Дата окончания действия сертификата/декларации': '25.05.2025', 'Высота упаковки': '13 см', 'Назначение обуви': 'повседневная', 'Цвет': 'черный', 'Особенности модели': 'Gore-Tex мембрана', 'Модель ботинок': 'ботинки', 'Дата регистрации сертификата/декларации': '26.05.2022', 'Декоративные элементы': 'без элементов', 'Номер сертификата соответствия': 'ЕАЭС RU С-DK.АЯ46.В.25234/22', 'Комплектация': 'Ботинки - 1 пара', 'Вид застежки': 'Шнурки', 'Полнота обуви (EUR)': 'F (6)', 'Ширина упаковки': '34 см', 'Пол':

Вопрос про цену - один из самых часто встречающихся. На бейзлайне модель не была уверена, к чему относитьь такие вопросы, так как слово "цена" фигурирует в карточках довольно часто. Намеренно включили в обучающую выборку такие моменты.

В данной ситуации модель учла, что состав формально описан в карточке, но по факту там его нет (указана только вода). Обучение произошло в результате часто встречающихся вопросов такого рода в обучающем сете.

In [13]:
product_id = int(input("Enter Product ID: "))
question = input("Enter Question: ")

result = inference_pipeline(product_id, question, collection, model, tokenizer)

print(f"Label: {result['label']} (Confidence: {result['confidence']:.2f})")
print(f"Description: {result['description']}")

Enter Product ID: 2025407
Enter Question: Когда товар появится?
Label: 0 (Confidence: 0.71)
Description: {'Упаковка': 'коробка', 'Комплектация': "краска-уход для волос L'Oreal Paris (Лореаль Париж) - 1 шт", 'Прямые поставки от производителя': 'да', 'Номер декларации соответствия': 'ЕАЭС N RU Д-BE.РА01.В.97110/21', 'Форма упаковки': 'без давления', 'Срок годности': '36 месяцев', 'Длина упаковки': '8 см', 'Дата регистрации сертификата/декларации': '28.05.2021', 'Раздел меню': 'Окрашивание волос и химическая завивка', 'Объем товара': '180 мл', 'Тип краски': 'полустойкая', 'Ширина упаковки': '9 см', 'Тон краски для волос': '1021, Светло-светло-русый перламутровый', 'Особенности краски для волос': 'безаммиачная', 'Высота упаковки': '17 см', 'Страна производства': 'Бельгия', 'Цвет': 'светло-русый', 'Состав': 'вода', 'Вес товара с упаковкой (г)': '239 г', 'Назначение косметического средства': 'для волос', 'Дата окончания действия сертификата/декларации': '27.05.2026'}  Краска для волос Castin

Изначально, до дообучения, модель была склонна относить вопросы о поставках к классу 1. В обучающую выборку включили множество вопросов в разных формулировках на такцю тему, отсюда довольно высокая уверенность.

In [14]:
product_id = int(input("Enter Product ID: "))
question = input("Enter Question: ")

result = inference_pipeline(product_id, question, collection, model, tokenizer)

print(f"Label: {result['label']} (Confidence: {result['confidence']:.2f})")
print(f"Description: {result['description']}")

Enter Product ID: 2025407
Enter Question: Седые волосы окрашивает?
Label: 1 (Confidence: 0.54)
Description: {'Упаковка': 'коробка', 'Комплектация': "краска-уход для волос L'Oreal Paris (Лореаль Париж) - 1 шт", 'Прямые поставки от производителя': 'да', 'Номер декларации соответствия': 'ЕАЭС N RU Д-BE.РА01.В.97110/21', 'Форма упаковки': 'без давления', 'Срок годности': '36 месяцев', 'Длина упаковки': '8 см', 'Дата регистрации сертификата/декларации': '28.05.2021', 'Раздел меню': 'Окрашивание волос и химическая завивка', 'Объем товара': '180 мл', 'Тип краски': 'полустойкая', 'Ширина упаковки': '9 см', 'Тон краски для волос': '1021, Светло-светло-русый перламутровый', 'Особенности краски для волос': 'безаммиачная', 'Высота упаковки': '17 см', 'Страна производства': 'Бельгия', 'Цвет': 'светло-русый', 'Состав': 'вода', 'Вес товара с упаковкой (г)': '239 г', 'Назначение косметического средства': 'для волос', 'Дата окончания действия сертификата/декларации': '27.05.2026'}  Краска для волос Cas

Уверенность невысокая, но все же ответ верный. Модель смущают синонимы + вопрос не очень похож на вопрос, короткий, требует, по сути, обещания.

In [15]:
product_id = int(input("Enter Product ID: "))
question = input("Enter Question: ")

result = inference_pipeline(product_id, question, collection, model, tokenizer)

print(f"Label: {result['label']} (Confidence: {result['confidence']:.2f})")
print(f"Description: {result['description']}")

Enter Product ID: 4803002
Enter Question: Здравствуйте! Подойдет для, проблемной кожи?Поры забивает?
Label: 1 (Confidence: 0.57)
Description: {'Состав': 'Вода,этилгексилметоксициннамат,каприловый/каприновый триглицерид,диметикон', 'Раздел меню': 'корейские бренды', 'Водостойкость средства': 'да', 'Срок годности': '30 месяцев', 'Ширина упаковки': '3 см', 'Страна производства': 'Россия', 'Комплектация': 'бб крем', 'Упаковка': 'картонная коробка', 'Длина упаковки': '5 см', 'Возрастные ограничения': '16+', 'Номер сертификата соответствия': 'RU.77.01.34.001.E.003032.12.16', 'Ставка НДС': '20', 'Дата регистрации сертификата/декларации': '13.12.2016', 'Высота упаковки': '16 см', 'Вес товара с упаковкой (г)': '62 г', 'SPF': 'SPF 10+', 'Вес товара без упаковки (г)': '42 г', 'Объем товара': '40 мл', 'Прямые поставки от производителя': 'да', 'Назначение косметического средства': 'для лица', 'Форма упаковки': 'без давления'}  Крем для лица BB идеальная кожа 10в1 с экстрактом розы 40 мл. Специально

In [17]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

model_path = "./fine_tuned_model"
model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)
test_data = pd.read_csv('/content/qa_card_test.csv')

def preprocess_and_tokenize(question, description):
    return tokenizer(question, description, return_tensors="pt", truncation=True, padding=True, max_length=512)

def infer(question, description, model, tokenizer):
    inputs = preprocess_and_tokenize(question, description)
    outputs = model(**{k: v.to(model.device) for k, v in inputs.items()})
    probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1).detach().cpu().numpy()
    predicted_label = probabilities.argmax(axis=1)[0]
    confidence = probabilities[0, predicted_label]
    return predicted_label, confidence

true_labels = []
predicted_labels = []
confidences = []

for _, row in test_data.iterrows():
    question = row["Question"]
    description = row["Description"]
    true_label = row["true_class"]

    predicted_label, confidence = infer(question, description, model, tokenizer)

    true_labels.append(true_label)
    predicted_labels.append(predicted_label)
    confidences.append(confidence)

accuracy = accuracy_score(true_labels, predicted_labels)
precision = precision_score(true_labels, predicted_labels)
recall = recall_score(true_labels, predicted_labels)
f1 = f1_score(true_labels, predicted_labels)
classification_rep = classification_report(true_labels, predicted_labels)

# Print results
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")




Accuracy: 0.8000
Precision: 1.0000
